# 08 - Retrain

Plan section 39: fresh, rewound, and teacher initializations compared on the same evolved genome.

Reproducibility contract (plan section 40): every session begins by
determining its configuration, locating and validating prior state,
and continues from there; it ends by saving metrics and a summary JSON.
Run inside a Kaggle session (or locally) after checking out this
repository, with the repository root as the working directory.


In [ ]:
import sys
from pathlib import Path

REPO = Path.cwd()
assert (REPO / 'src' / 'act').is_dir(), (
    f'run this notebook with the repository root as the working directory; '
    f'cwd was {REPO!r}'
)
sys.path.insert(0, str(REPO / 'src'))
print('python', sys.version.split()[0], '| repo', REPO)


# Fresh-initialization retraining (plan section 39)

The same evolved genome retrained from different initializations:
``fresh`` copies nothing, ``rewound`` restores a pre-mutation
checkpoint, and ``teacher`` transfers an existing model's
compatible tensors. Each policy reports its exact copied/skipped
parameter counts - the numbers the retraining comparisons record.


In [ ]:
from act.model import model_from_genome, rebuild_model_from_genome
from act.topology import create_genome, genome_hash, grow_expert
from act.training.retraining import initialize_retraining

genome = create_genome(42, hidden_size=24, vocab_size=32, depth=1,
                       ffn='moe', expert_count=4, active_experts=2)
base = model_from_genome(genome, context_size=12)
rewound_state = {k: v.clone() for k, v in base.state_dict().items()}
added = grow_expert(genome, genome.layers[0].id, step=12)
assert added is not None
evolved = rebuild_model_from_genome(base, genome, context_size=12)
print('evolved genome:', genome_hash(genome), '| added expert:', added)

In [ ]:
# Apply every policy to the same evolved model.
reports = {}
reports['fresh'] = initialize_retraining(evolved, mode='fresh')
reports['rewound'] = initialize_retraining(
    evolved, mode='rewound', rewind_state=rewound_state
)
reports['teacher'] = initialize_retraining(
    evolved, mode='teacher', source=base
)
assert reports['fresh'].copied_parameters == 0
# The grown expert has no pre-mutation source: rewound/teacher must
# skip exactly its tensors and copy the rest.
assert (
    reports['rewound'].copied_parameters
    == reports['teacher'].copied_parameters
    > 0
)
assert reports['rewound'].skipped_parameters > 0
for mode, report in reports.items():
    print(f"{mode:>8}: copied={report.copied_parameters} "
          f"skipped={report.skipped_parameters}")

In [ ]:
# Each initialized model trains under the shared runtime.
from act.training.runtime import (
    TrainingConfig,
    TrainingState,
    make_optimizer,
    train_steps,
)

train_config = TrainingConfig(sequence_length=12)
losses_by_mode = {}
for mode in ('fresh', 'rewound', 'teacher'):
    train_model = rebuild_model_from_genome(base, genome, context_size=12)
    if mode == 'rewound':
        initialize_retraining(
            train_model, mode='rewound', rewind_state=rewound_state
        )
    elif mode == 'teacher':
        initialize_retraining(train_model, mode='teacher', source=base)
    optimizer = make_optimizer(train_model, train_config)
    losses = train_steps(
        train_model, optimizer, TrainingState(), train_config, 8
    )
    losses_by_mode[mode] = losses[-1]
assert set(losses_by_mode) == {'fresh', 'rewound', 'teacher'}
for mode, final in losses_by_mode.items():
    print(f"{mode:>8}: final loss {final:.4f}")

In [ ]:
import json
from pathlib import Path

record_dir = Path('artifacts/notebook-records')
record_dir.mkdir(parents=True, exist_ok=True)
payload = {'notebook': '08_retrain',
 'evolved_hash': genome_hash(genome),
 'reports': {mode: {
     'copied': report.copied_parameters,
     'skipped': report.skipped_parameters,
 } for mode, report in reports.items()},
 'final_losses': losses_by_mode}
record = json.dumps(payload, sort_keys=True, indent=2) + '\n'
target = record_dir / '08-retrain.json'
existing = target.read_text() if target.exists() else None
if existing != record:
    target.write_text(record)
print('record:', target)